# Computer Vision — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/computer-vision/notebooks/computer-vision-lab.ipynb)

Companion notebook for the **Computer Vision** track (modules `cv-m1` … `cv-m19`).
Every section maps to one module and runs the code that module describes.

**Runs top to bottom on a fresh Colab runtime.** Parts 1–2 are CPU-only and take
seconds. Part 3 (`cv-m16`–`cv-m17`) downloads YOLO weights and trains a small
model — pick **Runtime → Change runtime type → T4 GPU** before running it, or
expect a few minutes on CPU.

Numbers that appear in the module text are re-derived here and checked with
`assert`, so if a cell runs silently the module's arithmetic held.

| Part | Modules | Needs |
|---|---|---|
| 1 — the image itself | m1–m7 | numpy, cv2, matplotlib, skimage |
| 2 — detection theory | m8–m11, m15, m18, m19 | numpy (+ torch for two cross-checks) |
| 3 — real detectors | m13, m16, m17 | torchvision, ultralytics, internet |

## Setup

Colab already ships NumPy, OpenCV, Matplotlib, scikit-image and PyTorch, so the
only install is `ultralytics` (used in Part 3). Sample images come from
`skimage.data` rather than a URL — no link to rot.

In [ ]:
# Colab has everything except ultralytics. Locally: pip install opencv-python
# scikit-image matplotlib torch torchvision ultralytics
!pip install -q ultralytics 2>/dev/null || echo "skip: install ultralytics manually if Part 3 fails"

import numpy as np, matplotlib.pyplot as plt
import cv2
from skimage import data

np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams['figure.figsize'] = (11, 3.2)
plt.rcParams['image.cmap'] = 'gray'

print('numpy', np.__version__, '| cv2', cv2.__version__)

In [ ]:
def show(*imgs, titles=None, cmap='gray', figsize=None):
    """Plot 1..n images side by side. RGB arrays are shown in colour,
    2-D arrays in greyscale. Used by every section below."""
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=figsize or (3.6 * n, 3.4))
    axes = np.atleast_1d(axes)
    for ax, im in zip(axes, imgs):
        ax.imshow(im, cmap=None if (im.ndim == 3) else cmap,
                  vmin=None if im.dtype != np.uint8 else 0,
                  vmax=None if im.dtype != np.uint8 else 255)
        ax.set_xticks([]); ax.set_yticks([])
    if titles:
        for ax, t in zip(axes, titles):
            ax.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()

def draw_boxes(img, boxes, labels=None, color=(0, 200, 0), thick=2):
    """boxes as xyxy in pixels. Returns a copy so the source is never mutated."""
    out = img.copy()
    if out.ndim == 2:
        out = cv2.cvtColor(out, cv2.COLOR_GRAY2RGB)
    for i, (x1, y1, x2, y2) in enumerate(np.asarray(boxes, dtype=int)):
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thick)
        if labels is not None:
            cv2.putText(out, str(labels[i]), (x1 + 2, max(12, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return out

GREY = data.camera()                 # 512x512 uint8
RGB  = data.astronaut()              # 512x512x3 uint8, RGB order
show(GREY, RGB, titles=['data.camera() — greyscale', 'data.astronaut() — RGB'])

---
# Part 1 — The image itself

## cv-m1 · Understanding digital images

An image is an array of integers. Shape, dtype and channel order are the three
facts that cause most downstream bugs.

In [ ]:
print('shape      ', GREY.shape, '  ← (H, W): rows first')
print('dtype      ', GREY.dtype)
print('range      ', GREY.min(), '..', GREY.max())
print('bytes      ', GREY.nbytes, f'({GREY.nbytes/1e6:.2f} MB)')
print('pixel [100,200] =', GREY[100, 200], ' ← row 100, column 200')

# The memory numeric from cv-m1: a 4000x3000 RGB photo as uint8, then as float32.
h, w, c = 4000, 3000, 3
u8  = h * w * c * 1
f32 = h * w * c * 4
print(f'\n4000x3000x3 uint8   = {u8/1e6:>7.1f} MB')
print(f'4000x3000x3 float32 = {f32/1e6:>7.1f} MB   ({f32/u8:.0f}x)')
print(f'a batch of 32 float32 = {32*f32/1e9:.2f} GB  ← resize BEFORE converting')
assert u8 == 36_000_000

In [ ]:
# BGR vs RGB. cv2.imread returns BGR; matplotlib expects RGB.
bgr = cv2.cvtColor(RGB, cv2.COLOR_RGB2BGR)      # simulate what imread hands you
show(RGB, bgr, titles=['correct (RGB)', 'the classic bug: BGR shown as RGB'])

# Greyscale is a weighted sum, not a mean. ITU-R BT.601 weights:
luma = cv2.cvtColor(RGB, cv2.COLOR_RGB2GRAY).astype(float)
mean = RGB.mean(axis=2)
manual = 0.299*RGB[...,0] + 0.587*RGB[...,1] + 0.114*RGB[...,2]

print('max |cv2 luma - manual| =', np.abs(luma - manual).max(), '(rounding only)')
print('max |luma - channel mean| =', np.abs(luma - mean).max(), '← they are NOT the same')
show(luma, mean, np.abs(luma-mean), titles=['0.299R+0.587G+0.114B', '(R+G+B)/3', 'difference'])

In [ ]:
# HSV separates colour identity from brightness. Note the uint8 hue range.
hsv = cv2.cvtColor(RGB, cv2.COLOR_RGB2HSV)
print('hue range in uint8 HSV:', hsv[...,0].min(), '..', hsv[...,0].max(), ' ← 0..179, not 0..359')
show(hsv[...,0], hsv[...,1], hsv[...,2], titles=['H (hue)', 'S (saturation)', 'V (value)'])

In [ ]:
# Quantisation: keep only 2**bits distinct levels. Sampling is untouched.
def quantise(img, bits):
    step = 256 // (2 ** bits)
    return ((img // step) * step).astype(np.uint8)

levels = [8, 4, 2, 1]
show(*[quantise(GREY, b) for b in levels], titles=[f'{b}-bit ({2**b} levels)' for b in levels])
for b in levels:
    print(f'{b}-bit -> {len(np.unique(quantise(GREY, b))):>3d} distinct values')

## cv-m2 · Math operations on images

Point operations: one input pixel, one output pixel. The only trap is what
`uint8` does on overflow — and NumPy and OpenCV disagree.

In [ ]:
a = np.uint8([[250]])
b = np.uint8([[10]])

print('numpy  a + b      =', (a + b).ravel()[0], ' ← wraps: 260 mod 256')
print('cv2.add(a, b)     =', cv2.add(a, b).ravel()[0], ' ← saturates')
safe = np.clip(a.astype(np.int16) + b.astype(np.int16), 0, 255).astype(np.uint8)
print('clip via int16    =', safe.ravel()[0])
assert (a + b).ravel()[0] == 4 and cv2.add(a, b).ravel()[0] == 255

In [ ]:
# Blend, brightness/contrast (affine), and gamma (non-linear, never clips).
blend = cv2.addWeighted(GREY, 0.6, cv2.resize(data.coins(), GREY.shape[::-1]), 0.4, 0)
linear = cv2.convertScaleAbs(GREY, alpha=1.5, beta=40)          # g = 1.5f + 40
gamma  = np.clip(255 * (GREY / 255) ** 0.5, 0, 255).astype(np.uint8)

show(GREY, blend, linear, gamma,
     titles=['original', 'blend a=0.6', 'g = 1.5f + 40 (clips)', 'gamma 0.5 (no clipping)'])

print('clipped to 255 by the linear map :', (linear == 255).mean() * 100, '% of pixels')
print('clipped to 255 by gamma          :', (gamma  == 255).mean() * 100, '% of pixels')
print('\ncv-m2 quiz: 1.5*200 + 40 =', 1.5*200+40, '-> saturates to',
      cv2.convertScaleAbs(np.uint8([[200]]), alpha=1.5, beta=40).ravel()[0])

In [ ]:
# Thresholding -> mask -> bitwise_and, and frame differencing for motion.
_, mask = cv2.threshold(GREY, 120, 255, cv2.THRESH_BINARY)
masked  = cv2.bitwise_and(GREY, GREY, mask=mask)

shifted = np.roll(GREY, 6, axis=1)                 # pretend the camera moved
motion  = cv2.absdiff(GREY, shifted)
_, motion_bin = cv2.threshold(motion, 25, 255, cv2.THRESH_BINARY)

show(mask, masked, motion, motion_bin,
     titles=['threshold(120)', 'bitwise_and', 'absdiff(t, t-1)', 'threshold(25)'])
print('Otsu picks the threshold for you:',
      cv2.threshold(GREY, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0])

## cv-m3 · Image filters

A filter replaces each pixel with a function of its neighbourhood. Read a linear
kernel by its **sum**: 1 preserves brightness, 0 is a derivative.

In [ ]:
KERNELS = {
    'identity':  np.array([[0,0,0],[0,1,0],[0,0,0]], float),
    'box blur':  np.ones((3,3)) / 9,
    'gaussian':  np.array([[1,2,1],[2,4,2],[1,2,1]], float) / 16,
    'sharpen':   np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], float),
    'sobel x':   np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float),
    'laplacian': np.array([[0,1,0],[1,-4,1],[0,1,0]], float),
}
for name, k in KERNELS.items():
    s = k.sum()
    kind = 'smoothing (brightness preserved)' if np.isclose(s, 1) else \
           'derivative (flat -> 0, signed output)' if np.isclose(s, 0) else 'scales brightness'
    print(f'{name:<10} sum = {s:>5.2f}   {kind}')

# A zero-sum kernel really does annihilate a flat region:
flat = np.full((32, 32), 128, np.uint8)
assert np.abs(cv2.filter2D(flat, cv2.CV_32F, KERNELS['sobel x'])).max() < 1e-6

In [ ]:
outs, titles = [], []
for name, k in KERNELS.items():
    r = cv2.filter2D(GREY, cv2.CV_32F, k)
    # signed responses need an offset to be visible
    outs.append(np.clip(r + 128, 0, 255).astype(np.uint8) if np.isclose(k.sum(), 0)
                else np.clip(r, 0, 255).astype(np.uint8))
    titles.append(name)
show(*outs[:3], titles=titles[:3]); show(*outs[3:], titles=titles[3:])

In [ ]:
# Separability: a 2-D Gaussian is rank 1, so it factors into two 1-D passes.
k = cv2.getGaussianKernel(15, -1)               # (15, 1) column
K2 = k @ k.T                                    # the full 15x15 kernel
print('rank of the 15x15 Gaussian:', np.linalg.matrix_rank(K2), ' <- rank 1 => separable')

naive, sep = 15**2, 15 + 15
H, W = 1080, 1920
print(f'naive     {naive:>3d} MACs/px  ->  {H*W*naive:,} MACs on {W}x{H}')
print(f'separable {sep:>3d} MACs/px  ->  {H*W*sep:,} MACs')
print(f'saving = {naive/sep:.1f}x')
assert H*W*naive == 466_560_000 and H*W*sep == 62_208_000 and naive/sep == 7.5

import time
big = cv2.resize(GREY, (1920, 1080))
t0 = time.perf_counter(); cv2.filter2D(big, -1, K2);          t_2d  = time.perf_counter()-t0
t0 = time.perf_counter(); cv2.sepFilter2D(big, -1, k, k);     t_sep = time.perf_counter()-t0
print(f'\nmeasured: 2-D {t_2d*1000:.1f} ms   separable {t_sep*1000:.1f} ms')

In [ ]:
# Median vs Gaussian on salt-and-pepper: the argument for rank filters.
rng = np.random.default_rng(0)
noisy = GREY.copy()
u = rng.random(GREY.shape)
noisy[u < 0.05] = 0
noisy[u > 0.95] = 255

gauss  = cv2.GaussianBlur(noisy, (5, 5), 0)
median = cv2.medianBlur(noisy, 5)
mae = lambda x: np.abs(x.astype(float) - GREY.astype(float)).mean()

show(GREY, noisy, gauss, median,
     titles=['clean', f'noisy (MAE {mae(noisy):.1f})',
             f'gaussian (MAE {mae(gauss):.1f})', f'median (MAE {mae(median):.1f})'])
assert mae(median) < mae(gauss), 'median should win on impulse noise'

In [ ]:
# Borders are a choice, and the choice is visible.
small = cv2.resize(GREY, (64, 64))
modes = [('zero', cv2.BORDER_CONSTANT), ('replicate', cv2.BORDER_REPLICATE),
         ('reflect', cv2.BORDER_REFLECT_101)]
big_k = np.ones((15, 15)) / 225
show(*[cv2.filter2D(small, -1, big_k, borderType=m) for _, m in modes],
     titles=[f'border: {n}' for n, _ in modes])

# Canny = blur -> gradient -> non-max suppression -> hysteresis.
# Step 3 is the same greedy rule NMS applies to boxes in cv-m10.
show(cv2.Canny(GREY, 50, 150), cv2.Canny(GREY, 150, 250),
     titles=['Canny(50, 150)', 'Canny(150, 250) — stricter hysteresis'])

## cv-m4 · Image transformations

One 3×3 matrix in homogeneous coordinates covers translate, rotate, scale,
shear and perspective. The bottom row is what separates affine from projective.

In [ ]:
def affine(tx=0, ty=0, deg=0, sx=1, sy=1, shx=0):
    """M = T . R . Shear . S  — the point is scaled first, translated last."""
    t = np.deg2rad(deg)
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]], float)
    R = np.array([[np.cos(t),-np.sin(t),0],[np.sin(t),np.cos(t),0],[0,0,1]], float)
    S = np.array([[sx,0,0],[0,sy,0],[0,0,1]], float)
    H = np.array([[1,shx,0],[0,1,0],[0,0,1]], float)
    return T @ R @ H @ S

# cv-m4 worked example: rotate (100, 50) by 30 degrees about the origin.
p = affine(deg=30) @ np.array([100, 50, 1.0])
print('rotated (100, 50) by 30 deg ->', p[:2].round(2))
assert np.allclose(p[:2], [61.60, 93.30], atol=0.01)

print('det(rotate+translate) =', round(np.linalg.det(affine(deg=47, tx=120)), 6), '<- rigid, area preserved')
print('det(scale 1.6, 0.7)   =', round(np.linalg.det(affine(sx=1.6, sy=0.7)), 6), '<- = the area factor')

In [ ]:
h, w = GREY.shape
M_rot   = cv2.getRotationMatrix2D((w/2, h/2), 30, 1.0)          # 2x3
M_shear = np.float32([[1, 0.35, 0], [0, 1, 0]])
show(GREY,
     cv2.warpAffine(GREY, M_rot,   (w, h)),
     cv2.warpAffine(GREY, M_shear, (w, h)),
     titles=['original', 'rotate 30 deg', 'shear 0.35'])

# A homography needs 4 point pairs: 8 unknowns, 2 equations each.
src = np.float32([[0,0],[w,0],[w,h],[0,h]])
dst = np.float32([[60,0],[w-30,40],[w,h],[0,h-50]])
Hm, _ = cv2.findHomography(src, dst)
print('bottom row of H =', Hm[2].round(6), '  <- non-zero => projective, not affine')
show(GREY, cv2.warpPerspective(GREY, Hm, (w, h)), titles=['original', 'homography (8 DoF)'])

In [ ]:
# Interpolation is not a free choice. Bilinear on a LABEL map invents classes.
labels = np.zeros((64, 64), np.uint8)
labels[10:30, 10:30] = 3
labels[34:56, 34:56] = 5

near = cv2.resize(labels, (256, 256), interpolation=cv2.INTER_NEAREST)
lin  = cv2.resize(labels, (256, 256), interpolation=cv2.INTER_LINEAR)
print('ids after INTER_NEAREST:', np.unique(near), ' <- only ids that existed')
print('ids after INTER_LINEAR :', np.unique(lin),  ' <- 1,2,4 were never annotated')
assert set(np.unique(near)) <= {0, 3, 5}
show(near, lin, titles=['INTER_NEAREST (correct for masks)', 'INTER_LINEAR (invented classes)'])

In [ ]:
# Why detection turns rotation augmentation off: the axis-aligned hull inflates.
def rotated_hull(bw, bh, deg):
    t = np.deg2rad(deg)
    return (bw*abs(np.cos(t)) + bh*abs(np.sin(t)),
            bw*abs(np.sin(t)) + bh*abs(np.cos(t)))

bw, bh = 100, 60
for d in (0, 10, 30, 45, 90):
    W_, H_ = rotated_hull(bw, bh, d)
    print(f'{d:>3d} deg -> {W_:6.1f} x {H_:6.1f} = {W_*H_:8.0f} px^2  '
          f'({W_*H_/(bw*bh)-1:+.0%} area)')
W_, H_ = rotated_hull(bw, bh, 30)
assert abs(W_ - 116.60) < 0.01 and abs(H_ - 101.96) < 0.01 and abs(W_*H_ - 11889) < 5

## cv-m5 · Image features

Edges slide along themselves; corners do not. Harris measures that with the
structure tensor, and the score `R = det(M) - k*trace(M)^2` avoids eigenvalues.

In [ ]:
gx = cv2.Sobel(GREY, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(GREY, cv2.CV_32F, 0, 1, ksize=3)
mag = cv2.magnitude(gx, gy)
ori = np.rad2deg(np.arctan2(gy, gx))
show(np.clip(gx+128,0,255).astype(np.uint8), np.clip(gy+128,0,255).astype(np.uint8),
     (255*mag/mag.max()).astype(np.uint8),
     titles=['Gx', 'Gy', '||grad|| = sqrt(Gx^2+Gy^2)'])

In [ ]:
# The three cases, straight from R's sign. k = 0.04 as in the module.
def harris_R(l1, l2, k=0.04):
    return l1*l2 - k*(l1+l2)**2

for name, (l1, l2) in {'flat': (0.01, 0.01), 'edge': (1.0, 0.0),
                       'corner': (1.0, 1.0), 'cv-m5 quiz': (0.9, 0.02)}.items():
    print(f'{name:<12} l1={l1:<5} l2={l2:<5} R = {harris_R(l1, l2):+.4f}')
assert harris_R(0.9, 0.02) < 0            # an edge, not a corner
assert harris_R(1.2, 0.9, k=0.05) > 0.85  # cv-quiz value

R = cv2.cornerHarris(GREY.astype(np.float32), blockSize=3, ksize=3, k=0.04)
pts = np.argwhere(R > 0.02 * R.max())
vis = cv2.cvtColor(GREY, cv2.COLOR_GRAY2RGB)
for y, x in pts:
    cv2.circle(vis, (int(x), int(y)), 2, (0, 255, 0), -1)
print(f'\n{len(pts)} corner pixels above 2% of R_max')
show(np.clip(R/R.max()*255, 0, 255).astype(np.uint8), vis,
     titles=['Harris response R', 'R > 2% of max'])

In [ ]:
# Detector + descriptor + matching, the classical pipeline in five lines.
orb = cv2.ORB_create(600)
img2 = cv2.warpAffine(GREY, cv2.getRotationMatrix2D((256,256), 25, 0.9), (512,512))

k1, d1 = orb.detectAndCompute(GREY, None)
k2, d2 = orb.detectAndCompute(img2, None)
matches = sorted(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(d1, d2),
                 key=lambda m: m.distance)
print(f'{len(k1)} + {len(k2)} keypoints, {len(matches)} mutual matches')
print('ORB descriptor is', d1.shape[1]*8, 'bits -> Hamming distance, not L2')

vis = cv2.drawMatches(GREY, k1, img2, k2, matches[:40], None, flags=2)
plt.figure(figsize=(11, 5)); plt.imshow(vis); plt.axis('off')
plt.title('ORB matches across a 25 deg rotation + 0.9 scale'); plt.show()

In [ ]:
# HOG dimensionality for the classic 64x128 pedestrian window (cv-m5).
win, cell, block, bins = (64, 128), 8, 2, 9
cells = (win[0]//cell, win[1]//cell)                 # 8 x 16
blocks = (cells[0]-block+1, cells[1]-block+1)        # 7 x 15
dim = blocks[0]*blocks[1] * block*block * bins
print(f'cells  {cells[0]} x {cells[1]}')
print(f'blocks {blocks[0]} x {blocks[1]}  (stride 1 cell, so they overlap)')
print(f'dim    {blocks[0]*blocks[1]} blocks x {block*block*bins} = {dim}')
assert dim == 3780

# Cross-check against a library. skimage is used rather than cv2.HOGDescriptor
# because the latter is missing from OpenCV 5 and from some headless builds.
from skimage.feature import hog as sk_hog
window = cv2.resize(GREY, (64, 128))          # cv2.resize takes (width, height)
v = sk_hog(window, orientations=bins, pixels_per_cell=(cell, cell),
           cells_per_block=(block, block), block_norm='L2-Hys')
print('skimage agrees:', v.shape)
assert v.shape[0] == dim

## cv-m6 · Regions of interest

An ROI is an array slice. The three box formats are where detection bugs hide,
because converting them wrongly never raises.

In [ ]:
roi = GREY[120:300, 180:380]                  # [y1:y2, x1:x2] — rows first
print('roi.shape', roi.shape, '  base is the parent array:', roi.base is not None)

copy = GREY.copy()
view = copy[0:40, 0:40]
view[:] = 255                                  # writes THROUGH to `copy`
print('slice is a view -> parent changed:', copy[0, 0] == 255)

# Out-of-range slices are silently clipped, so a bad box yields a smaller ROI.
print('GREY[0:5000, 0:5000].shape =', GREY[0:5000, 0:5000].shape, '  <- no error raised')

mask = np.zeros_like(GREY); mask[120:300, 180:380] = 255
show(roi, cv2.bitwise_and(GREY, GREY, mask=mask),
     titles=[f'crop {roi.shape}', 'mask (shape preserved)'])

In [ ]:
# The three formats, with a round-trip assertion.
def xywh_to_xyxy(b): x, y, w, h = b; return (x, y, x+w, y+h)
def xyxy_to_cxcywh(b): x1, y1, x2, y2 = b; return ((x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1)
def cxcywh_to_xyxy(b): cx, cy, w, h = b; return (cx-w/2, cy-h/2, cx+w/2, cy+h/2)

box = (120, 80, 60, 90)                                  # xywh, the cv-m6 quiz
xyxy = xywh_to_xyxy(box); cxcywh = xyxy_to_cxcywh(xyxy)
print('xywh  ', box)
print('xyxy  ', xyxy)
print('cxcywh', cxcywh)
assert cxcywh == (150, 125, 60, 90)
assert cxcywh_to_xyxy(cxcywh) == xyxy                     # round trip

# Predicted boxes routinely fall outside the frame. Clip explicitly.
H, W = 480, 640
pred = np.array([-12, 40, 655, 500], float)
clipped = np.array([max(0,pred[0]), max(0,pred[1]), min(W,pred[2]), min(H,pred[3])])
area = lambda b: (b[2]-b[0])*(b[3]-b[1])
print(f'\nbefore clip {pred} area {area(pred):,.0f}')
print(f'after  clip {clipped} area {area(clipped):,.0f}  ({area(clipped)/area(pred)-1:+.1%})')
assert abs(area(clipped)/area(pred) - 1 + 0.082) < 0.001

In [ ]:
# Boxes without a neural network: colour range -> morphology -> contours.
hsv = cv2.cvtColor(RGB, cv2.COLOR_RGB2HSV)
m = cv2.inRange(hsv, (0, 60, 60), (25, 255, 255))          # warm/skin-ish range
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  k)                # kill specks
m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)                # fill pinholes

cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
boxes = [cv2.boundingRect(c) for c in cnts if cv2.contourArea(c) > 400]
print(f'{len(cnts)} contours, {len(boxes)} above the area filter')
show(m, draw_boxes(RGB, [xywh_to_xyxy(b) for b in boxes]),
     titles=['cleaned mask', 'boundingRect of each contour'])

## cv-m7 · Image distributions

A histogram discards position entirely. Equalisation is for viewing;
**fixed-statistic normalisation** is for training.

In [ ]:
low = cv2.convertScaleAbs(GREY, alpha=0.25, beta=96)       # squash into a narrow band
eq   = cv2.equalizeHist(low)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(low)

fig, ax = plt.subplots(2, 3, figsize=(11, 5))
for j, (im, name) in enumerate([(low, 'low contrast'), (eq, 'equalizeHist'), (clahe, 'CLAHE')]):
    ax[0, j].imshow(im, cmap='gray', vmin=0, vmax=255); ax[0, j].set_title(name, fontsize=10)
    ax[0, j].set_xticks([]); ax[0, j].set_yticks([])
    ax[1, j].hist(im.ravel(), bins=64, range=(0, 255), color='#15803d')
    ax[1, j].set_yticks([])
plt.tight_layout(); plt.show()

for im, name in [(low, 'low contrast'), (eq, 'equalised'), (clahe, 'CLAHE')]:
    print(f'{name:<14} min {im.min():>3d}  max {im.max():>3d}  '
          f'mean {im.mean():>6.1f}  std {im.std():>5.1f}')
assert eq.std() > low.std() * 3

In [ ]:
# A histogram cannot see structure: shuffle the pixels, keep the histogram.
shuf = GREY.ravel().copy(); np.random.default_rng(0).shuffle(shuf)
shuf = shuf.reshape(GREY.shape)
same = np.array_equal(np.bincount(GREY.ravel(), minlength=256),
                      np.bincount(shuf.ravel(), minlength=256))
print('identical histograms:', same)
show(GREY, shuf, titles=['original', 'same histogram, no structure'])
assert same

In [ ]:
# One-pass dataset statistics — the numbers that go into transforms.Normalize.
def dataset_stats(images):
    n = 0; s = np.zeros(3); s2 = np.zeros(3)
    for im in images:
        x = im.reshape(-1, 3).astype(np.float64) / 255.0
        n += x.shape[0]; s += x.sum(0); s2 += (x**2).sum(0)
    mean = s / n
    return mean, np.sqrt(np.maximum(s2/n - mean**2, 0))

imgs = [RGB, data.coffee(), data.chelsea()]
mean, std = dataset_stats(imgs)
print('mean', mean.round(4))
print('std ', std.round(4))
print('\ntransforms.Normalize(mean=%s, std=%s)' % (list(mean.round(3)), list(std.round(3))))
print('ImageNet defaults for comparison: mean [0.485 0.456 0.406] std [0.229 0.224 0.225]')

# The identity is exact — cross-check against the two-pass computation.
allpx = np.concatenate([im.reshape(-1,3) for im in imgs]).astype(np.float64)/255
assert np.allclose(mean, allpx.mean(0)) and np.allclose(std, allpx.std(0))

---
# Part 2 — Detection, from scratch

Everything in this part is plain NumPy so you can read the mechanism rather
than an API. Two cells cross-check against `torchvision.ops`.

## cv-m8 · IoU

In [ ]:
def to_xyxy(b):
    x, y, w, h = b
    return np.array([x, y, x+w, y+h], float)

def iou(a, b):
    """a, b in xyxy. The max(0, .) is what stops disjoint boxes reporting
    a spurious positive overlap from two negative side lengths."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2-x1) * max(0.0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def giou(a, b):
    cx1, cy1 = min(a[0], b[0]), min(a[1], b[1])
    cx2, cy2 = max(a[2], b[2]), max(a[3], b[3])
    c = (cx2-cx1)*(cy2-cy1)
    inter_x = max(0.0, min(a[2],b[2])-max(a[0],b[0]))
    inter_y = max(0.0, min(a[3],b[3])-max(a[1],b[1]))
    inter = inter_x*inter_y
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return iou(a, b) - (c-u)/c

# The cv-m8 worked example, step by step.
A, B = to_xyxy((50, 40, 100, 120)), to_xyxy((90, 70, 100, 120))
inter = (min(A[2],B[2])-max(A[0],B[0])) * (min(A[3],B[3])-max(A[1],B[1]))
union = 12000 + 12000 - inter
print(f'intersection = {min(A[2],B[2])-max(A[0],B[0]):.0f} x '
      f'{min(A[3],B[3])-max(A[1],B[1]):.0f} = {inter:.0f}')
print(f'union        = 12000 + 12000 - {inter:.0f} = {union:.0f}')
print(f'IoU          = {inter:.0f}/{union:.0f} = {iou(A,B):.4f}   '
      f'-> at threshold 0.5 this is a FALSE POSITIVE')
assert inter == 5400 and union == 18600 and abs(iou(A, B) - 0.2903) < 1e-4

In [ ]:
# IoU flatlines at 0 for disjoint boxes; GIoU keeps a gradient.
base = to_xyxy((0, 0, 40, 40))
print(f'{"gap":>5} {"IoU":>8} {"GIoU":>9}')
for gap in (0, 20, 60, 150, 400):
    other = to_xyxy((40+gap, 0, 40, 40))
    print(f'{gap:>5} {iou(base, other):>8.4f} {giou(base, other):>9.4f}')
assert iou(base, to_xyxy((100,0,40,40))) == 0 and giou(base, to_xyxy((100,0,40,40))) < 0

# Same prediction, different verdicts, purely from the reporting threshold.
for t in (0.25, 0.5, 0.75):
    print(f'@IoU {t}: {"TP" if iou(A,B) >= t else "FP"}')

## cv-m9 · The localisation loss

`L = L_obj + 1[object] * (lambda * L_box + L_cls)`. The indicator is the part
people drop, and dropping it trains the box head on background.

In [ ]:
def smooth_l1(x, beta=1.0):
    a = np.abs(x)
    return np.where(a < beta, 0.5*x*x/beta, a - 0.5*beta)

x = np.linspace(-6, 6, 601)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(x, 0.5*x**2, label='L2  x^2/2', color='#dc2626')
ax[0].plot(x, smooth_l1(x), label='smooth L1', color='#15803d')
ax[0].plot(x, np.abs(x),   label='L1  |x|',  color='#2563eb')
ax[0].set_ylim(0, 8); ax[0].set_title('loss'); ax[0].legend(fontsize=8)
ax[1].plot(x, x, color='#dc2626'); ax[1].plot(x, np.clip(x, -1, 1), color='#15803d')
ax[1].plot(x, np.sign(x), color='#2563eb')
ax[1].set_ylim(-3, 3); ax[1].set_title('gradient  (smooth L1 is capped at 1)')
plt.tight_layout(); plt.show()

# Continuity and differentiability at the transition:
assert np.isclose(smooth_l1(1-1e-9), smooth_l1(1+1e-9))
assert np.isclose(smooth_l1(5), 4.5) and np.isclose(0.5*5**2, 12.5)

errs = np.array([0.02, 0.03, 0.01, 5.0])       # three fine, one mislabelled box
print('L2        total %.4f   outlier share %.4f%%' % ((0.5*errs**2).sum(), 100*0.5*25/(0.5*errs**2).sum()))
print('smoothL1  total %.4f   outlier share %.4f%%' % (smooth_l1(errs).sum(), 100*4.5/smooth_l1(errs).sum()))

In [ ]:
def localisation_loss(p_obj, box_pred, cls_probs, present, box_gt, cls_gt, lam=5.0):
    """The composite loss with the indicator that masks background regions."""
    eps = 1e-9
    t = 1.0 if present else 0.0
    l_obj = -(t*np.log(p_obj+eps) + (1-t)*np.log(1-p_obj+eps))
    if not present:
        return l_obj, 0.0, 0.0, l_obj              # box and class terms MASKED OUT
    l_box = smooth_l1(np.asarray(box_pred) - np.asarray(box_gt)).sum()
    l_cls = -np.log(cls_probs[cls_gt] + eps)
    return l_obj, l_box, l_cls, l_obj + lam*l_box + l_cls

gt, pred = [0.30, 0.32, 0.38, 0.55], [0.34, 0.36, 0.35, 0.50]
probs = np.array([0.15, 0.70, 0.15])

for present in (True, False):
    o, b, c, tot = localisation_loss(0.82, pred, probs, present, gt, 1)
    tag = 'object' if present else 'background'
    print(f'{tag:<11} L_obj {o:.4f}  L_box {b:.4f}  L_cls {c:.4f}  total {tot:.4f}')
_, b_bg, c_bg, _ = localisation_loss(0.82, pred, probs, False, gt, 1)
assert b_bg == 0.0 and c_bg == 0.0, 'background must contribute no box/class gradient'

## cv-m10 · NMS and mAP

In [ ]:
def nms(boxes, scores, thr=0.5):
    """Greedy: take the best, delete everything overlapping it, repeat."""
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order):
        i = order[0]; keep.append(i)
        rest = order[1:]
        if not len(rest): break
        ious = np.array([iou(boxes[i], boxes[j]) for j in rest])
        order = rest[ious <= thr]
    return keep

# Three objects, six detections — A/B/C duplicate one object, E/F are TWO
# occluding objects at IoU 0.556, G is a lone false positive.
B_ = np.array([[46,44,142,162],[58,54,154,172],[38,36,146,166],
               [226,56,310,168],[250,56,334,168],[150,150,212,216]], float)
S_ = np.array([0.94, 0.88, 0.79, 0.83, 0.71, 0.35])
ids = list('ABCEFG')

print('IoU(A,B) = %.3f   IoU(A,C) = %.3f   IoU(E,F) = %.3f'
      % (iou(B_[0],B_[1]), iou(B_[0],B_[2]), iou(B_[3],B_[4])))
for t in (0.5, 0.6, 0.7, 0.9):
    kept = [ids[i] for i in nms(B_, S_, t)]
    print(f'thr {t}: kept {kept}')
print('\n0.50 deletes F, a REAL object. 0.70 keeps F but keeps duplicate B.')
print('Only 0.60 is right for this scene, and nothing told you that in advance.')
assert [ids[i] for i in nms(B_, S_, 0.5)] == ['A', 'E', 'G']
assert [ids[i] for i in nms(B_, S_, 0.6)] == ['A', 'E', 'F', 'G']

In [ ]:
# Cross-check the hand-written NMS against torchvision's.
import torch
from torchvision.ops import nms as tv_nms
tv = sorted(tv_nms(torch.tensor(B_, dtype=torch.float32),
                   torch.tensor(S_, dtype=torch.float32), 0.5).tolist())
print('ours      ', sorted(nms(B_, S_, 0.5)))
print('torchvision', tv)
assert sorted(nms(B_, S_, 0.5)) == tv

In [ ]:
def pr_curve(scores, tps, n_gt):
    """All-point (VOC2010 / COCO) AP: area under the monotone envelope."""
    order = np.argsort(scores)[::-1]
    tp = np.cumsum(np.asarray(tps, float)[order])
    fp = np.cumsum(1 - np.asarray(tps, float)[order])
    recall = tp / n_gt
    precision = tp / (tp + fp)
    env = np.maximum.accumulate(precision[::-1])[::-1]     # best precision at recall >= r
    ap = float(np.sum(np.diff(np.concatenate([[0], recall])) * env))
    return recall, precision, env, ap

scores = [0.95, 0.91, 0.88, 0.80, 0.74, 0.68, 0.55, 0.41]
tps    = [1,    1,    0,    1,    0,    1,    0,    0]
r, p, env, ap = pr_curve(scores, tps, n_gt=5)

print(f'{"#":>2} {"score":>6} {"TP":>3} {"recall":>7} {"prec":>6} {"env":>6}')
for i in range(len(scores)):
    print(f'{i+1:>2} {scores[i]:>6.2f} {tps[i]:>3} {r[i]:>7.2f} {p[i]:>6.3f} {env[i]:>6.3f}')
print(f'\nAP = {ap:.4f}   (recall caps at {r[-1]:.1f} — one object was never found)')
assert abs(ap - 0.6833) < 1e-4

plt.figure(figsize=(4.2, 3.4))
plt.step(np.concatenate([[0], r]), np.concatenate([[env[0]], env]), where='post', color='#15803d')
plt.fill_between(np.concatenate([[0], r]), np.concatenate([[env[0]], env]),
                 step='post', alpha=0.2, color='#15803d')
plt.scatter(r, p, s=18, c=['#15803d' if t else '#dc2626' for t in tps], zorder=3)
plt.xlim(0, 1); plt.ylim(0, 1.05); plt.xlabel('recall'); plt.ylabel('precision')
plt.title(f'AP = {ap:.4f}'); plt.tight_layout(); plt.show()

## cv-m11 · Fully convolutional networks

A dense layer *is* a convolution whose kernel matches its input. Proving it
takes five lines, and it is the whole reason detection stopped cropping.

In [ ]:
import torch, torch.nn as nn

C, n, K = 512, 7, 4096
fc   = nn.Linear(n*n*C, K)
conv = nn.Conv2d(C, K, kernel_size=n)
with torch.no_grad():
    conv.weight.copy_(fc.weight.view(K, C, n, n))
    conv.bias.copy_(fc.bias)

x = torch.randn(1, C, n, n)
with torch.no_grad():
    a = fc(x.flatten(1))
    b = conv(x).flatten(1)
print('max |dense - conv| =', (a - b).abs().max().item())
assert torch.allclose(a, b, atol=1e-4), 'the two layers must be the same function'

# The difference only shows on a LARGER input: the dense layer cannot run at all,
# the conv slides and emits a map with one column per window position.
big = torch.randn(1, C, 11, 9)
with torch.no_grad():
    print('conv on a 11x9 map ->', tuple(conv(big).shape), ' <- a score MAP, not a vector')
try:
    fc(big.flatten(1))
except RuntimeError as e:
    print('dense on the same input ->', str(e).split('(')[0].strip())

In [ ]:
# The cv-m11 cost arithmetic: the saving IS the window overlap.
def window_count(size, win, stride, levels=1, factor=1.5):
    n, s = 0, float(size)
    for _ in range(levels):
        k = max(0, int((s - win) // stride) + 1)
        n += k*k; s /= factor
    return n

size, win, stride = 512, 64, 8
crops = window_count(size, win, stride)
fcn   = (size/win) ** 2                     # conv cost scales with input area
print(f'{crops:,} crops vs {fcn:.0f} window-forwards for one conv pass')
print(f'speed-up {crops/fcn:.1f}x   (window overlap at this stride: {100*(1-stride/win):.0f}%)')
assert crops == 3249 and fcn == 64

print(f'\n{"stride":>7} {"crops":>9} {"FCN":>6} {"speed-up":>9}')
for s in (16, 8, 4, 2):
    c = window_count(size, win, s)
    print(f'{s:>7} {c:>9,} {fcn:>6.0f} {c/fcn:>8.1f}x')
print('\nHalving the stride quadruples the crops and leaves the FCN cost untouched.')

---
# Part 3 — Real detectors

Needs internet (weights) and is much happier on a GPU.
**Runtime → Change runtime type → T4 GPU.**

## cv-m13 · Faster R-CNN (two-stage)

In [ ]:
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).eval().to(dev)
names = weights.meta['categories']

x = torch.from_numpy(RGB).permute(2, 0, 1).float().div(255).to(dev)   # (3,H,W) in [0,1]
with torch.no_grad():
    out = model([x])[0]

keep = out['scores'] > 0.5
boxes = out['boxes'][keep].cpu().numpy()
labs  = [f"{names[i]} {s:.2f}" for i, s in zip(out['labels'][keep].cpu(), out['scores'][keep].cpu())]
print(f'{len(out["boxes"])} raw detections -> {keep.sum().item()} above 0.5')
for l in labs: print(' ', l)
show(draw_boxes(RGB, boxes, labs), titles=['Faster R-CNN (already NMS-ed)'], figsize=(6, 6))

## cv-m15 · The YOLO output tensor

Before running a real YOLO, work out what it emits. `S x S x (B*5 + C)`.

In [ ]:
def yolo_tensor(S, B, Cn):
    return dict(cells=S*S, per_cell=B*5+Cn, total=S*S*(B*5+Cn), boxes=S*S*B)

for S, B, Cn, tag in [(7, 2, 20, 'YOLO v1 (VOC)'), (13, 5, 80, 'v2-style'), (19, 3, 10, 'cv-quiz')]:
    t = yolo_tensor(S, B, Cn)
    print(f'{tag:<16} S={S:<3} B={B} C={Cn:<3} -> {S}x{S}x{t["per_cell"]} '
          f'= {t["total"]:,} numbers, {t["boxes"]} boxes')
assert yolo_tensor(7,2,20)['total'] == 1470
assert yolo_tensor(13,5,80)['total'] == 17745 and yolo_tensor(19,3,10)['total'] == 9025

def assign_cell(box_xywh, S, img=1.0):
    """Which cell owns the box, and the offsets the network must regress."""
    x, y, w, h = box_xywh
    cx, cy = (x + w/2)/img, (y + h/2)/img
    col, row = min(S-1, int(cx*S)), min(S-1, int(cy*S))
    return dict(row=row, col=col, cx=cx, cy=cy,
                tx=cx*S - col, ty=cy*S - row, tw=w/img, th=h/img)

a = assign_cell((30, 120, 110, 120), S=7, img=280)
print('\ndog at (30,120,110,120) on a 280px image, S=7:')
for k, v in a.items(): print(f'  {k:<4} {v if isinstance(v,int) else round(v,4)}')
assert 0 <= a['tx'] < 1 and 0 <= a['ty'] < 1, 'tx/ty are offsets INSIDE the cell'

## cv-m16 · YOLO inference and reading the output

In [ ]:
from ultralytics import YOLO
import cv2, os

ROOT = '/content/cvlab' if os.path.isdir('/content') else 'cvlab'
os.makedirs(ROOT, exist_ok=True)

cv2.imwrite(f'{ROOT}/sample.jpg', cv2.cvtColor(RGB, cv2.COLOR_RGB2BGR))

model = YOLO('yolov8n.pt')                       # downloads ~6 MB on first use
res = model(f'{ROOT}/sample.jpg', conf=0.25, iou=0.45, imgsz=640, verbose=False)[0]

print('boxes.xyxy  ', tuple(res.boxes.xyxy.shape), '<- absolute pixels, ORIGINAL image')
print('boxes.conf  ', tuple(res.boxes.conf.shape))
print('boxes.cls   ', tuple(res.boxes.cls.shape), '\n')
for (x1,y1,x2,y2), c, k in zip(res.boxes.xyxy.cpu().numpy(),
                               res.boxes.conf.cpu().numpy(),
                               res.boxes.cls.cpu().numpy().astype(int)):
    print(f'{res.names[k]:<14} conf={c:.3f}  xywh=({x1:.0f},{y1:.0f},{x2-x1:.0f},{y2-y1:.0f})')

show(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB), titles=['YOLOv8n'], figsize=(6, 6))

In [ ]:
# conf and iou are POST-PROCESSING knobs: no retraining needed to change them.
print(f'{"conf":>6} {"iou":>6} {"detections":>11}')
for conf in (0.05, 0.25, 0.60):
    for iou_t in (0.30, 0.70):
        n = len(model(f'{ROOT}/sample.jpg', conf=conf, iou=iou_t, verbose=False)[0].boxes)
        print(f'{conf:>6.2f} {iou_t:>6.2f} {n:>11d}')
print('\nRaising conf   -> fewer, more certain boxes (precision up, recall down).')
print('Raising iou    -> LESS suppression, so MORE overlapping boxes survive.')

In [ ]:
# Letterboxing: resize preserving aspect ratio, then pad to a square.
def letterbox_params(h, w, imgsz=640):
    r = min(imgsz/h, imgsz/w)
    nh, nw = round(h*r), round(w*r)
    return r, (imgsz-nh)/2, (imgsz-nw)/2          # scale, pad_y each side, pad_x each side

for (h, w) in [(720, 1280), (600, 800), (1080, 1920)]:
    r, py, px = letterbox_params(h, w)
    print(f'{w}x{h} @640 -> scale {r:.3f}, pad y {py:.0f} px each side, pad x {px:.0f}')
r, py, px = letterbox_params(720, 1280)
assert abs(r-0.5) < 1e-9 and py == 140 and px == 0
r, py, px = letterbox_params(600, 800)
assert abs(r-0.8) < 1e-9 and py == 80

print('\nMapping a box back:  x_orig = (x_letterbox - pad_x) / scale')
print('Forget the pad and every box is shifted; forget the scale and every box is half size.')
print('Nothing errors either way — this is the classic "my ONNX boxes moved" bug.')

## cv-m17 · Training YOLO on your own data

A synthetic two-class dataset (circles and rectangles) so the whole loop runs
in a couple of minutes with no download. The **folder convention and the label
format are exactly what a real dataset needs.**

In [ ]:
import os, cv2, numpy as np, shutil
from pathlib import Path

DS = Path(ROOT) / 'shapes'
if DS.exists(): shutil.rmtree(DS)
for split in ('train', 'val'):
    (DS/'images'/split).mkdir(parents=True, exist_ok=True)
    (DS/'labels'/split).mkdir(parents=True, exist_ok=True)

SZ, rng = 320, np.random.default_rng(7)

def make_sample(path_img, path_lbl):
    img = np.full((SZ, SZ, 3), rng.integers(30, 90), np.uint8)
    img = cv2.add(img, rng.integers(0, 25, (SZ, SZ, 3), dtype=np.uint8))   # texture
    lines = []
    for _ in range(rng.integers(1, 4)):
        cls = int(rng.integers(0, 2))
        r = int(rng.integers(24, 52))
        cx, cy = int(rng.integers(r+4, SZ-r-4)), int(rng.integers(r+4, SZ-r-4))
        col = tuple(int(v) for v in rng.integers(140, 255, 3))
        if cls == 0:
            cv2.circle(img, (cx, cy), r, col, -1)
            w = h = 2*r
        else:
            w, h = int(r*rng.uniform(1.2, 2.0)), int(r*rng.uniform(0.8, 1.4))
            cv2.rectangle(img, (cx-w//2, cy-h//2), (cx+w//2, cy+h//2), col, -1)
        # class cx cy w h — all normalised to [0, 1]
        lines.append(f'{cls} {cx/SZ:.6f} {cy/SZ:.6f} {min(w,SZ)/SZ:.6f} {min(h,SZ)/SZ:.6f}')
    cv2.imwrite(str(path_img), img)
    path_lbl.write_text('\n'.join(lines))

for split, n in (('train', 80), ('val', 20)):
    for i in range(n):
        make_sample(DS/'images'/split/f'{split}_{i:03d}.jpg',
                    DS/'labels'/split/f'{split}_{i:03d}.txt')

(DS/'data.yaml').write_text(f"""path: {DS}
train: images/train
val: images/val
names:
  0: circle
  1: rect
""")
print(open(DS/'data.yaml').read())
print('example label file:'); print((DS/'labels'/'train'/'train_000.txt').read_text())

In [ ]:
# ALWAYS draw a few labels back onto their images before training.
# Thirty seconds here catches all four of the cv-m17 label bugs.
def preview(stem, split='train'):
    img = cv2.cvtColor(cv2.imread(str(DS/'images'/split/f'{stem}.jpg')), cv2.COLOR_BGR2RGB)
    for line in (DS/'labels'/split/f'{stem}.txt').read_text().splitlines():
        c, cx, cy, w, h = line.split()
        cx, cy, w, h = (float(v)*SZ for v in (cx, cy, w, h))
        p1 = (int(cx-w/2), int(cy-h/2)); p2 = (int(cx+w/2), int(cy+h/2))
        cv2.rectangle(img, p1, p2, (0, 255, 0), 2)
        cv2.putText(img, ['circle','rect'][int(c)], (p1[0], max(12, p1[1]-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1)
    return img

show(*[preview(f'train_{i:03d}') for i in range(4)],
     titles=['labels drawn back onto the image']*4)

# Sanity checks a converter should always run:
for f in (DS/'labels'/'train').glob('*.txt'):
    for line in f.read_text().splitlines():
        parts = line.split()
        assert len(parts) == 5, f'{f.name}: expected 5 fields'
        assert 0 <= int(parts[0]) <= 1, f'{f.name}: class id out of range'
        assert all(0.0 <= float(v) <= 1.0 for v in parts[1:]), f'{f.name}: not normalised'
print('all labels: 5 fields, class in range, values in [0,1]')

In [ ]:
# Train. ~1 min on a T4, a few minutes on CPU. Start from COCO weights.
model = YOLO('yolov8n.pt')
results = model.train(
    data=str(DS/'data.yaml'),
    epochs=25, imgsz=SZ, batch=16, patience=10,
    degrees=0.0,          # rotation inflates axis-aligned boxes — see cv-m4
    mosaic=1.0, close_mosaic=5,
    project=ROOT, name='shapes', exist_ok=True, verbose=False, plots=True,
)
m = model.val(data=str(DS/'data.yaml'), verbose=False)
print(f'\nmAP@50    = {m.box.map50:.4f}')
print(f'mAP@50-95 = {m.box.map:.4f}')
print(f'per class : {dict(zip(["circle","rect"], m.box.maps.round(3)))}')

In [ ]:
# Read the artefacts, not just the number.
from IPython.display import Image, display
run = Path(ROOT)/'shapes'
for f in ('results.png', 'confusion_matrix.png', 'PR_curve.png', 'val_batch0_pred.jpg'):
    if (run/f).exists():
        print(f); display(Image(filename=str(run/f), width=560))
print('The background column of the confusion matrix is the one to read first:')
print('  leakage INTO background  = missed detections (low recall)')
print('  leakage FROM background  = false positives')
print('Those two need opposite fixes, and mAP alone cannot tell them apart.')

In [ ]:
# Predict with the fine-tuned weights.
best = run/'weights'/'best.pt'
tuned = YOLO(str(best))
imgs = sorted((DS/'images'/'val').glob('*.jpg'))[:4]
outs = [cv2.cvtColor(tuned(str(p), conf=0.35, verbose=False)[0].plot(), cv2.COLOR_BGR2RGB)
        for p in imgs]
show(*outs, titles=['fine-tuned predictions']*4)

# Export stops BEFORE NMS — you get the raw (1, 4+C, N) head tensor.
# tuned.export(format='onnx', imgsz=SZ, simplify=True)

## cv-m18 · Anchor boxes

Anchors are priors on shape. They are chosen by k-means with **`d = 1 - IoU`**,
not Euclidean distance, so large boxes cannot dominate the clustering.

In [ ]:
def shape_iou(a, b):
    """IoU of two boxes sharing a centre — only shape matters."""
    inter = np.minimum(a[...,0], b[...,0]) * np.minimum(a[...,1], b[...,1])
    return inter / (a[...,0]*a[...,1] + b[...,0]*b[...,1] - inter)

def anchor_kmeans(boxes, k, iters=100):
    # Deterministic init: sort by area and take k evenly spaced boxes. Random
    # init drops into bad local minima often enough to make the mean-IoU curve
    # look non-monotone, which is an artefact of the seed, not of k.
    order = boxes[np.argsort(boxes[:, 0] * boxes[:, 1])]
    cent = order[((np.arange(k) + 0.5) / k * len(order)).astype(int)].copy()
    prev = None
    for _ in range(iters):
        d = 1 - shape_iou(boxes[:, None, :], cent[None, :, :])   # (N, k)
        a = d.argmin(1)
        if prev is not None and (a == prev).all(): break
        prev = a
        for j in range(k):
            if (a == j).any(): cent[j] = boxes[a == j].mean(0)
    return cent[np.argsort(cent[:,0]*cent[:,1])], a, shape_iou(boxes, cent[a]).mean()

# Use the real (w, h) from the cv-m17 dataset if Part 3 has been run; otherwise
# synthesise three shape families so this cell stands on its own.
try:
    wh = np.array([[float(v) for v in l.split()[3:5]]
                   for f in (DS/'labels'/'train').glob('*.txt')
                   for l in f.read_text().splitlines()])
    src_tag = 'cv-m17 synthetic-shapes dataset'
except NameError:
    r = np.random.default_rng(23)
    wh = np.concatenate([
        r.normal([0.11, 0.34], 0.035, (26, 2)),    # pedestrians — tall and thin
        r.normal([0.40, 0.20], 0.075, (24, 2)),    # vehicles — wide
        r.normal([0.09, 0.09], 0.025, (18, 2)),    # signs — small squares
    ]).clip(0.02, 0.95)
    src_tag = 'synthetic 3-family box set'
print(f'{len(wh)} ground-truth boxes from the {src_tag}')

print(f'\n{"k":>2} {"mean IoU":>9} {"gain":>7}')
prev_iou = 0
for k in range(1, 8):
    cent, assign, m_iou = anchor_kmeans(wh, k)
    print(f'{k:>2} {m_iou:>9.4f} {m_iou-prev_iou:>+7.4f}')
    prev_iou = m_iou
cent, assign, m_iou = anchor_kmeans(wh, 5)
print('\n5 anchors (w, h):'); print(cent.round(4))
print('\nThe knee is where k matches the number of real shape families.')
print('The curve can dip by a hair: the centroid update is an arithmetic mean,')
print('which does not exactly minimise the 1-IoU objective, so Lloyd\'s algorithm')
print('is not guaranteed monotone here the way Euclidean k-means is.')
assert anchor_kmeans(wh, 3)[2] > anchor_kmeans(wh, 1)[2], 'more anchors must not hurt'

In [ ]:
# Why NOT Euclidean: the same relative mismatch costs 100x more on a large box.
small = (np.array([0.02, 0.03]), np.array([0.04, 0.06]))    # 2x mismatch, tiny
large = (np.array([0.20, 0.30]), np.array([0.40, 0.60]))    # 2x mismatch, big
for tag, (a, b) in [('small pair', small), ('large pair', large)]:
    print(f'{tag}: euclidean {np.linalg.norm(a-b):.4f}   1-IoU {1-shape_iou(a, b):.4f}')
print('\nEuclidean punishes the large pair ~10x harder for the SAME shape error,')
print('so the centroids migrate to large boxes. 1-IoU is scale-free.')
assert np.isclose(shape_iou(*small), shape_iou(*large)), '1-IoU must be scale invariant'

plt.figure(figsize=(4.4, 3.8))
plt.scatter(wh[:,0], wh[:,1], s=10, c=assign, cmap='viridis', alpha=0.6)
plt.scatter(cent[:,0], cent[:,1], marker='s', s=90, facecolors='none', edgecolors='r', lw=2)
plt.xlabel('width (normalised)'); plt.ylabel('height (normalised)')
plt.title(f'k=5 anchors, mean IoU {m_iou:.3f}'); plt.tight_layout(); plt.show()

In [ ]:
# YOLOv2's constrained decode: sigmoid on the centre, exp on the size.
sigmoid = lambda z: 1/(1+np.exp(-z))

def decode(tx, ty, tw, th, cx, cy, pw, ph, S=13):
    bx, by = sigmoid(tx)+cx, sigmoid(ty)+cy       # can NEVER leave the cell
    bw, bh = pw*np.exp(tw), ph*np.exp(th)         # always positive, symmetric in log
    return bx, by, bw, bh, bx/S, by/S, bw/S, bh/S

bx, by, bw, bh, *frac = decode(0.4, -1.2, 0.25, -0.1, cx=4, cy=6, pw=1.87, ph=3.34)
print(f'sigma(0.4)={sigmoid(0.4):.3f}  sigma(-1.2)={sigmoid(-1.2):.3f}')
print(f'grid units : bx {bx:.3f}  by {by:.3f}  bw {bw:.3f}  bh {bh:.3f}')
print(f'image frac : ({frac[0]:.3f}, {frac[1]:.3f}) size ({frac[2]:.3f}, {frac[3]:.3f})')
print(f'\nty = -1.2 did NOT push the box out of row 6 — sigma clamped it to {sigmoid(-1.2):.3f}.')
assert 4 <= bx < 5 and 6 <= by < 7
assert abs(bx-4.599) < 1e-3 and abs(bw-2.401) < 1e-3

# The imbalance anchors create: YOLOv3 @416 over three scales.
pos = 13**2 + 26**2 + 52**2
print(f'\npositions {13**2}+{26**2}+{52**2} = {pos:,}  x3 anchors = {pos*3:,} boxes/image')
print(f'~7 objects x 3 positive anchors = 21 positives -> 1 : {(pos*3-21)//21} imbalance')
assert pos == 3549 and pos*3 == 10647

## cv-m19 · SSD default boxes

Six feature maps, decreasing resolution. Scale is the map's job; aspect ratio
is the default box's.

In [ ]:
SSD = [('conv4_3', 38, 4), ('conv7', 19, 6), ('conv8_2', 10, 6),
       ('conv9_2', 5, 6), ('conv10_2', 3, 4), ('conv11_2', 1, 4)]

s_min, s_max, m = 0.2, 0.9, 6
scales = [s_min + (s_max-s_min)*(k)/(m-1) for k in range(m)]

total = 0
print(f'{"layer":<10} {"map":>6} {"k/cell":>7} {"s_k":>6} {"boxes":>8} {"running":>9}')
for (name, sz, k), s in zip(SSD, scales):
    n = sz*sz*k; total += n
    print(f'{name:<10} {sz:>4}^2 {k:>7} {s:>6.2f} {n:>8,} {total:>9,}')
print(f'{"TOTAL":<10} {"":>6} {"":>7} {"":>6} {total:>8,}')
assert total == 8732
assert [round(s, 2) for s in scales] == [0.20, 0.34, 0.48, 0.62, 0.76, 0.90]

print(f'\nSSD300 {total:,} boxes vs YOLO v1 7*7*2 = {7*7*2}  -> {total/98:.0f}x more')
print(f'{100*5776/total:.0f}% of them sit on conv4_3, the highest-resolution map:')
print('small objects are the hard case and need the fine grid.')

matched = 10
print(f'\n{matched}/{total} boxes match an object = {100*matched/total:.2f}% positive.')
print('That is why SSD needs hard negative mining at 3:1 — without it the')
print('background term outweighs the positives by ~1000x and nothing trains.')

In [ ]:
# Aspect ratios at one scale: area is held constant, only shape varies.
def default_boxes(s, ratios=(1, 2, 3, 0.5, 1/3)):
    return [(s*np.sqrt(a), s/np.sqrt(a)) for a in ratios]

for w, h in default_boxes(0.48):
    print(f'w {w:.4f}  h {h:.4f}  ratio {w/h:>5.2f}  area {w*h:.4f}')
assert all(abs(w*h - 0.48**2) < 1e-12 for w, h in default_boxes(0.48))
print('\nEvery box has area s^2 — scale is set by the feature map, shape by the ratio.')

fig, ax = plt.subplots(1, len(scales), figsize=(13, 2.4))
for a, (s, (name, sz, k)) in zip(ax, zip(scales, SSD)):
    for w, h in default_boxes(s)[:k]:
        a.add_patch(plt.Rectangle((0.5-w/2, 0.5-h/2), w, h, fill=False, ec='#15803d', lw=1.2))
    a.set_xlim(0,1); a.set_ylim(0,1); a.set_xticks([]); a.set_yticks([])
    a.set_title(f'{name}\n{sz}$^2$, s={s:.2f}', fontsize=8)
plt.tight_layout(); plt.show()

---
## Where to go next

- **FPN** — a top-down pathway so every resolution has deep semantics. Swap
  `fasterrcnn_resnet50_fpn` for a non-FPN backbone and compare small-object AP.
- **Focal loss / RetinaNet** — `torchvision.models.detection.retinanet_resnet50_fpn`.
- **Anchor-free** — YOLOv8 above already is; try FCOS via
  `torchvision.models.detection.fcos_resnet50_fpn`.
- **DETR** — no anchors and no NMS; duplicates are handled by bipartite matching
  in the loss instead of in post-processing.

Re-run the Part 2 cells with your own boxes and scores — every function there is
short enough to modify in place, which is the point of having them in NumPy.